# MagmaBallz — Kaggle Full SOLO Hard-100 Benchmark (structural vs nhatminh)

> **Target:** `benchmarks/solo_hard_100.json` (100 problems: 50×evaluation_extra_hard, 20×evaluation_hard, 30×evaluation_order5, all 300s) vs 2 solvers:
> - `my_submission/solo_variants/structural_cache.py` (id + structural cache)
> - `examples/solo/demos/nhatminh.py` (ETP oracle + OpenRouter `openai/gpt-oss-120b`)

This notebook is **fully self-contained for Kaggle** — it installs Lean/ lake/ Mathlib, builds the judge, installs Python deps, runs the full benchmark with resume, and aggregates results. Enable **Internet** in Kaggle notebook settings before running.

Expected runtime: ~50 min per solver @ `workers=4` (100×300s/4 + Lean overhead) → ~1.7h for both. Kaggle limit = 12h. Resume is on, so interrupted runs pick up where they left off.


## 0 — Kaggle checklist

- Settings → **Internet ON** (needed for `elan`, `lake exe cache get`, `pip install openai`)
- Settings → **Persist outputs** ON if you want results under `/kaggle/working`
- Secrets → add `OPENROUTER_API_KEY` or `OPENAI_API_KEY` if you want LLM calls (solvers fall back to deterministic search otherwise)
- Accelerator: **None/CPU** is fine (Lean is CPU-bound, 4 cores → `workers=4`)


In [ ]:
# ── 0.1 Detect Kaggle env + set paths (robust to papermill cwd) ──
import os, sys, pathlib, json, subprocess, shutil, textwrap
from pathlib import Path

def find_repo_root() -> Path:
    # 1. Check cwd and parents for repo
    cwd = Path.cwd().resolve()
    for cand in [cwd] + list(cwd.parents)[:6]:
        if (cand / "lean-toolchain").exists() and (cand / "lakefile.lean").exists():
            return cand
        if (cand / "MagmaBallz" / "lean-toolchain").exists():
            return (cand / "MagmaBallz").resolve()
    # 2. Check known locations
    for base in [Path("/home/anhtu77/Coding/MagmaBallz"), Path("/kaggle/working/MagmaBallz"), Path("/tmp/MagmaBallz"), Path("/tmp/MagmaBallz_clone")]:
        if (base / "lean-toolchain").exists() and (base / "lakefile.lean").exists():
            return base.resolve()
    # 3. Brute-force find under /kaggle and /home (limited)
    for search_root in ["/kaggle", "/home"]:
        try:
            out = subprocess.check_output(["find", str(search_root), "-maxdepth", "5", "-name", "lean-toolchain", "-type", "f"], text=True, timeout=5)
            for line in out.splitlines():
                cand = Path(line.strip()).parent.resolve()
                if (cand / "lakefile.lean").exists():
                    return cand
        except: pass
    # 4. Fallback: if on Kaggle, working dir is intended clone parent, not repo itself
    if Path("/kaggle").exists():
        # return a path that will be used as clone target, not the empty working dir
        return Path("/kaggle/working").resolve()
    return cwd

IS_KAGGLE = Path("/kaggle").exists()
print(f"IS_KAGGLE={IS_KAGGLE}")
print(f"python={sys.version}")
print(f"initial cwd={Path.cwd()}")
try:
    print(f"initial ls={list(Path.cwd().iterdir())[:15]}")
except: print("ls failed")

REPO_ROOT = find_repo_root()
print(f"REPO_ROOT found: {REPO_ROOT}  exists lean-toolchain={(REPO_ROOT / 'lean-toolchain').exists()}")
# Only chdir if REPO_ROOT actually contains repo
if (REPO_ROOT / "lean-toolchain").exists():
    try:
        os.chdir(REPO_ROOT)
        print(f"chdir to {REPO_ROOT} -> cwd={Path.cwd()}")
        get_ipython().run_line_magic("cd", str(REPO_ROOT))
    except Exception as e:
        print(f"chdir failed: {e}")
else:
    print(f"REPO_ROOT does not contain repo yet, staying in {Path.cwd()} - will clone in next cell")

print(f"REPO_ROOT={REPO_ROOT}")
print(f"cwd after={Path.cwd()}")


In [ ]:
# ── 0.2 System check + Python deps ──
!python3 --version
!pip --version
!df -h | head -20
!nproc; free -h | head -5

# Install Python deps (idempotent)
!pip install -q openai tqdm pandas matplotlib
import openai, tqdm, pandas
print(f"openai={openai.__version__}")

In [ ]:
# ── 0.3 Ensure repo (uses REPO_ROOT from 0.1) — clones prototype/morellm if needed ──
from pathlib import Path
import os, subprocess, shutil

print(f"REPO_ROOT={REPO_ROOT}  cwd={Path.cwd()}")

# Helper: does path contain repo?
def has_repo(p: Path) -> bool:
    return (p / "lakefile.lean").exists() and (p / "lean-toolchain").exists()

# 1. If REPO_ROOT already has repo, done
if has_repo(REPO_ROOT):
    print(f"Repo verified at {REPO_ROOT}")
else:
    # Try alternative locations before cloning
    alt_candidates = [
        Path.cwd() / "MagmaBallz",
        Path("/kaggle/working/MagmaBallz"),
        Path("/tmp/MagmaBallz"),
        Path("/tmp/MagmaBallz_clone"),
        Path("/home/anhtu77/Coding/MagmaBallz"),
    ]
    found = None
    for cand in alt_candidates:
        if has_repo(cand):
            REPO_ROOT = cand.resolve()
            found = cand
            print(f"Found repo at {REPO_ROOT}")
            break
    # Kaggle dataset search
    if not has_repo(REPO_ROOT) and Path("/kaggle/input").exists():
        print("Kaggle input detected, searching for dataset...")
        try:
            out = subprocess.check_output(["find","/kaggle/input","-name","lean-toolchain"], text=True, timeout=10)
            print(out)
            for line in out.splitlines():
                cand = Path(line.strip()).parent.resolve()
                if has_repo(cand):
                    REPO_ROOT = cand
                    print(f"Found dataset repo at {REPO_ROOT}")
                    break
        except Exception as e:
            print(f"find failed: {e}")

    # Still not found -> clone
    if not has_repo(REPO_ROOT):
        # Choose clone target: on Kaggle use /kaggle/working/MagmaBallz (persisted), else /tmp
        if Path("/kaggle").exists():
            clone_target = Path("/kaggle/working/MagmaBallz")
        else:
            clone_target = Path("/tmp/MagmaBallz_clone")
        print(f"Cloning from https://github.com/IamKrill1n/MagmaBallz.git branch prototype/morellm to {clone_target} ...")
        # Clean previous stale clone
        if clone_target.exists():
            print(f"Removing stale {clone_target}")
            get_ipython().system(f'rm -rf "{clone_target}" 2>&1 | tail -5')
        # Clone (need internet ON)
        ret = get_ipython().system(f'git clone --branch prototype/morellm --single-branch https://github.com/IamKrill1n/MagmaBallz.git "{clone_target}" 2>&1 | tail -20')
        # Check
        if has_repo(clone_target):
            REPO_ROOT = clone_target.resolve()
            print(f"Clone succeeded, REPO_ROOT={REPO_ROOT}")
        else:
            print(f"Clone failed — check Internet ON, URL/branch, or Kaggle dataset mount")
            print(f"ls /tmp: {list(Path('/tmp').iterdir())[:10] if Path('/tmp').exists() else 'no tmp'}")
            if Path("/kaggle/working").exists():
                print(f"ls /kaggle/working: {list(Path('/kaggle/working').iterdir())[:20]}")
            # Try alternative fallback /tmp
            alt_tmp = Path("/tmp/MagmaBallz_clone")
            if has_repo(alt_tmp):
                REPO_ROOT = alt_tmp.resolve()
                print(f"Fallback to {REPO_ROOT}")
            else:
                print("ERROR: No repo found after clone. Aborting.")
                print("Try: enable Internet in Kaggle Settings, or upload repo as Dataset to /kaggle/input")

# Ensure correct branch if git repo
if (REPO_ROOT / ".git").exists():
    print(f"Git repo at {REPO_ROOT}, ensuring branch prototype/morellm...")
    get_ipython().system(f'cd "{REPO_ROOT}" && git rev-parse --abbrev-ref HEAD 2>&1 | head -5')
    get_ipython().system(f'cd "{REPO_ROOT}" && git fetch origin prototype/morellm 2>&1 | tail -10')
    get_ipython().system(f'cd "{REPO_ROOT}" && git checkout prototype/morellm 2>&1 | tail -10')
    get_ipython().system(f'cd "{REPO_ROOT}" && git log --oneline -3 2>&1 | head -10')
else:
    print(f"REPO_ROOT {REPO_ROOT} is not a git repo (may be ok if copied)")

# Final chdir and verify
os.chdir(REPO_ROOT)
try:
    get_ipython().run_line_magic("cd", str(REPO_ROOT))
except: pass
print(f"FINAL REPO_ROOT={Path.cwd()}")
get_ipython().system('pwd; ls -lh | head -30')
get_ipython().system(f'cat "{REPO_ROOT}/lean-toolchain"')
get_ipython().system(f'cd "{REPO_ROOT}" && git rev-parse --abbrev-ref HEAD 2>&1 | head -5')
get_ipython().system(f'cat "{REPO_ROOT}/benchmarks/solo_hard_100.json" | head -70')

# Fail fast if still not found
if not has_repo(REPO_ROOT):
    raise FileNotFoundError(f"REPO_ROOT {REPO_ROOT} still missing lean-toolchain after all attempts — check Internet/Dataset")


## 1 — Lean toolchain setup (full, Kaggle-safe, idempotent)

Mirrors `scripts/setup.sh:1` — elan → toolchain → lake update → cache get → build judge modules. Re-running is safe (skips installed parts).

In [ ]:
# ── 1.1 Install elan if missing ──
import shutil, subprocess, os
from pathlib import Path
!which elan || echo "elan not found"
!elan --version 2>&1 | head -5

if not shutil.which("elan"):
    print("Installing elan...")
    !curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain none
    # Update PATH for this notebook kernel
    os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ["PATH"]
    !export PATH="$HOME/.elan/bin:$PATH" && elan --version
else:
    print("elan already installed")
    os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ.get("PATH","")
    !elan --version
    !which lean; which lake || true

In [ ]:
# ── 1.2 Install pinned toolchain + set default (REPO_ROOT-aware) ──
from pathlib import Path
import os
# Ensure in REPO_ROOT
os.chdir(REPO_ROOT)
try: get_ipython().run_line_magic("cd", str(REPO_ROOT))
except: pass

toolchain_path = REPO_ROOT / "lean-toolchain"
print(f"toolchain_path={toolchain_path} exists={toolchain_path.exists()} cwd={Path.cwd()}")
toolchain = toolchain_path.read_text().strip()
print(f"Required toolchain: {toolchain}")
get_ipython().system('elan toolchain list 2>&1 | head -20')
# Use shell with REPO_ROOT env
get_ipython().system('elan toolchain install "$toolchain" 2>&1 | tail -20')
get_ipython().system('elan default "$toolchain"')
get_ipython().system('lean --version')
get_ipython().system('lake --version')


In [ ]:
# ── 1.3 Fetch Mathlib + build judge (heavy, cached) ──
# This is the slow part (~2GB cache, 5-10min with cache, 1h+ without).
import os
from pathlib import Path
os.chdir(REPO_ROOT)
try: get_ipython().run_line_magic("cd", str(REPO_ROOT))
except: pass
os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ["PATH"]

# Show lake manifest pinned rev
get_ipython().system('cat lake-manifest.json | head -30')
print("Running lake update...")
get_ipython().system('lake update 2>&1 | tail -30')

print("Downloading Mathlib cache...")
get_ipython().system('lake exe cache get 2>&1 | tail -30')

print("Building judge modules (JudgeMagma, JudgeDecide, JudgeFinOp, JudgeSupport)...")
get_ipython().system('lake build JudgeMagma.Magma JudgeDecide.DecideBang JudgeFinOp.MemoFinOp JudgeSupport.Inspect 2>&1 | tail -50')
print("Lake build done")
get_ipython().system('ls -lh .lake/build/lib/Judge* 2>&1 | head -20')


In [ ]:
# ── 1.4 Write .env.judge + export to notebook env (REPO_ROOT-aware) ──
import subprocess, pathlib, os, shutil
from pathlib import Path
os.chdir(REPO_ROOT)
try: get_ipython().run_line_magic("cd", str(REPO_ROOT))
except: pass
# Find lean/lake
lean_bin = shutil.which("lean") or str(Path.home() / ".elan" / "bin" / "lean")
lake_bin = shutil.which("lake") or str(Path.home() / ".elan" / "bin" / "lake")
get_ipython().system('which lean')
get_ipython().system('which lake')
lean_bin = shutil.which("lean")
lake_bin = shutil.which("lake")
print(f"LEAN_BIN={lean_bin}")
print(f"LAKE_BIN={lake_bin}")

env_file = REPO_ROOT / ".env.judge"
env_file.write_text(f'''# Auto-generated by Kaggle notebook
export LEAN_BIN="{lean_bin}"
export LAKE_BIN="{lake_bin}"
export PATH="$HOME/.elan/bin:$PATH"
''')
print(env_file.read_text())
os.environ["LEAN_BIN"] = lean_bin
os.environ["LAKE_BIN"] = lake_bin
os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ["PATH"]
print("Env exported")


In [ ]:
# ── 1.5 Smoke test judge (must be accepted) ──
import json, sys, os
from pathlib import Path
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
from judge.verify import verify_answer, JudgeConfig
lean_bin = Path(os.environ["LEAN_BIN"])
lake_bin = Path(os.environ["LAKE_BIN"])
config = JudgeConfig(lake_bin=lake_bin, lean_bin=lean_bin)
problem = json.loads((REPO_ROOT / "tests/fixtures/problems/p_true_basic.json").read_text())
answer = (REPO_ROOT / "tests/fixtures/answers/accepted_true_basic.answer.json").read_text()
result = verify_answer(problem, answer, config=config)
print(json.dumps({"status": result["status"]}, indent=2))
assert result["status"] == "accepted", f"Smoke failed: {result}"
print("✅ Judge smoke PASSED")

# Also run harness quick check (optional, ~30s)
get_ipython().system('python3 scripts/run_harness.py 2>&1 | tail -100')


## 2 — Kaggle secrets (LLM)

Solvers work without LLM (deterministic fallback), but `my_submission/solver.py` and `nhatminh.py` can use `openai/gpt-oss-120b` via OpenRouter. Add key as Kaggle Secret.

In [ ]:
import os
from pathlib import Path
# Kaggle Secrets injection: Settings → Secrets → Add Secret (OPENROUTER_API_KEY)
# In Kaggle, secrets are auto-exported if you attach them. We also try manual fallback.
print(f"OPENROUTER_API_KEY set: {bool(os.environ.get('OPENROUTER_API_KEY'))}")
print(f"OPENAI_API_KEY set: {bool(os.environ.get('OPENAI_API_KEY'))}")
print(f"OPENAI_BASE_URL={os.environ.get('OPENAI_BASE_URL','(default https://openrouter.ai/api/v1)')}")

# Show pipeline llm config
import subprocess
print((REPO_ROOT / "pipeline/config.json").read_text()[:800])

# If no key, benchmark still runs — llm calls will return error and solver continues deterministically
if not os.environ.get("OPENROUTER_API_KEY") and not os.environ.get("OPENAI_API_KEY"):
    print("⚠️  No LLM key — solvers will run deterministic-only (expected, still valid)")
else:
    print("✅ LLM key present")

# ── Force OpenRouter config for nhatminh (overrides pipeline/config.json at runtime) ──
import json
cfg_path = REPO_ROOT / "pipeline/config.json"
cfg = json.loads(cfg_path.read_text())
cfg["llm"]["model"] = "openai/gpt-oss-120b"
cfg["llm"]["provider"] = "deepinfra/bf16"
cfg["llm"]["base_url"] = "https://openrouter.ai/api/v1"
cfg["llm"]["api_key_env"] = "OPENROUTER_API_KEY"
cfg["llm"]["reasoning_effort"] = "low"
cfg["llm"]["max_output_tokens"] = 65536
# Write back for run_one_submission (it reloads via load_config)
# Keep original as backup
import shutil
shutil.copy(cfg_path, cfg_path.with_suffix(".bak"))
cfg_path.write_text(json.dumps(cfg, indent=2))
print("Patched pipeline/config.json for OpenRouter:")
print(cfg_path.read_text()[:600])
# Ensure env points to openrouter
if not os.environ.get("OPENROUTER_API_KEY") and os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = os.environ["OPENAI_API_KEY"]
    print("Mirrored OPENAI_API_KEY -> OPENROUTER_API_KEY")


## 3 — Load profile + submissions

`benchmarks/solo_hard_100.json:1` defines 100 problems, 300s each, workers=8. On Kaggle we use `workers=4` (Kaggle CPU=4) and keep 300s.

In [ ]:
import json, hashlib, copy, platform, sys, time, pathlib, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
from pipeline.proxy import load_config, load_problems
from pipeline.solo_benchmark import load_profile, discover_problem_files, selected_problem_entries, timeout_for, _benchmark_fingerprint, _sha256, _summarize

PROFILE_PATH = REPO_ROOT / "benchmarks" / "solo_hard_100.json"
BASE_CONFIG_PATH = REPO_ROOT / "pipeline" / "config.json"

profile = load_profile(PROFILE_PATH)
base_config = load_config(BASE_CONFIG_PATH)
print(f"Profile: {profile['name']}  mode={profile['mode']}  submission={profile['submission']}")
print(f"Timeouts: {profile['timeouts_seconds']}")
print(f"Output dir: {profile['output_directory']}")
print(f"Selections: {len(profile['problems']['selections'])} files")
for sel in profile["problems"]["selections"]:
    print(f"  {sel['path']}: {len(sel['ids'])} ids")

# Submissions to bench (2) — file submissions are staged to temp dir/solver.py by solo_benchmark
SUBMISSIONS = {
    "structural": REPO_ROOT / "my_submission" / "solo_variants" / "structural_cache.py",
    "nhatminh": REPO_ROOT / "examples" / "solo" / "demos" / "nhatminh.py",
}  # only 2: structural_cache (deterministic) + nhatminh (OpenRouter)
for k, p in SUBMISSIONS.items():
    print(f"{k:12s} -> {p}  exists={p.exists()}  size={p.stat().st_size if p.exists() else 'MISSING'}")

# Kaggle workers override — 4 is sweet spot for 4 vCPUs + Lean contention
KAGGLE_WORKERS = 4
print(f"KAGGLE_WORKERS={KAGGLE_WORKERS} (profile workers={profile['execution']['workers']})")

# Discover jobs
problem_root = (REPO_ROOT / profile["problems"].get("root", "")).resolve()
problem_files = discover_problem_files(profile)
print(f"Problem files: {len(problem_files)}")
from dataclasses import dataclass
@dataclass(frozen=True)
class Job:
    source_path: Path
    source_relative: Path
    source_index: int
    problem: dict
    timeout_seconds: int
jobs = []
for sp in problem_files:
    rel = sp.relative_to(problem_root)
    for idx, prob in selected_problem_entries(sp, profile):
        jobs.append(Job(sp, rel, idx, prob, timeout_for(prob, rel, profile)))
print(f"Total jobs: {len(jobs)}")
from collections import Counter
print(Counter(j.timeout_seconds for j in jobs))
jobs[:2]


In [ ]:
# ── 3.1 Dry-run sanity (no solving, just validation) ──
import subprocess, sys, os
from pathlib import Path
os.chdir(REPO_ROOT)
# Use the canonical solo_benchmark dry-run for each submission
for name, sub in SUBMISSIONS.items():
    print(f"\n=== dry-run {name}: {sub} ===")
    get_ipython().system('python3 -m pipeline.solo_benchmark --profile benchmarks/solo_hard_100.json --submission "{sub}" --dry-run 2>&1 | tail -20')


## 4 — Full benchmark runner (Kaggle, resume, progress)

Re-implements `pipeline/solo_benchmark.py:406` inside notebook so progress renders inline and output goes to `/kaggle/working` (persisted). Resume is automatic via `pipeline/results/solo_hard_100/<name>-<hash>/`.

In [ ]:
import os; os.chdir(REPO_ROOT); import sys; sys.path.insert(0, str(REPO_ROOT))
import copy, json, hashlib, shutil, tempfile, time
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
import sys
sys.path.insert(0, ".")
from pipeline.proxy import load_config, run_solver
from pipeline.solo_benchmark import _result_path, _load_completed, _append_result, _atomic_write_json, _benchmark_fingerprint

# Ensure output under /kaggle/working if on Kaggle (persisted), else repo pipeline/results
import os
if Path("/kaggle/working").exists():
    # Mirror repo results to kaggle working so they survive kernel restart
    kaggle_out = Path("/kaggle/working") / "solo_hard_100_results"
    kaggle_out.mkdir(parents=True, exist_ok=True)
    # Also keep repo path
    repo_out = Path("pipeline/results/solo_hard_100")
    repo_out.mkdir(parents=True, exist_ok=True)
    print(f"Kaggle mode: results mirrored to {kaggle_out} and {repo_out}")
else:
    print(f"Local mode: {Path('pipeline/results/solo_hard_100').resolve()}")

def run_one_submission(submission_path: Path, profile_path: Path, workers: int = 4):
    from pipeline.solo_benchmark import load_profile, discover_problem_files, selected_problem_entries, timeout_for
    profile = load_profile(profile_path)
    base_config = load_config(REPO_ROOT / profile["base_pipeline_config"])
    problem_root = (REPO_ROOT / profile["problems"].get("root", "")).resolve()
    problem_files = discover_problem_files(profile)
    # Build jobs
    jobs = []
    for sp in problem_files:
        rel = sp.relative_to(problem_root)
        for idx, prob in selected_problem_entries(sp, profile):
            jobs.append((sp, rel, idx, prob, timeout_for(prob, rel, profile)))
    # Fingerprint + staging (file submissions → temp dir/solver.py)
    solver_path = submission_path if submission_path.is_file() else submission_path / "solver.py"
    assert solver_path.exists(), f"solver not found: {solver_path}"
    staging = None
    if submission_path.is_file():
        staging = tempfile.TemporaryDirectory(prefix="magmaballz-solo-")
        staged = Path(staging.name)
        shutil.copyfile(solver_path, staged / "solver.py")
        submission = staged
    else:
        submission = submission_path
    fingerprint = _benchmark_fingerprint(profile_path.resolve(), profile, submission_path.resolve(), (REPO_ROOT / profile["base_pipeline_config"]).resolve(), problem_files)
    submission_name = submission_path.stem  # nhatminh, solver, structural, etc.
    output_root = REPO_ROOT / profile["output_directory"]
    run_dir = output_root / f"{submission_name}-{fingerprint[:12]}"
    run_dir.mkdir(parents=True, exist_ok=True)
    # Kaggle mirror
    if Path("/kaggle/working").exists():
        kaggle_mirror = Path("/kaggle/working") / "solo_hard_100_results" / f"{submission_name}-{fingerprint[:12]}"
        kaggle_mirror.mkdir(parents=True, exist_ok=True)
    else:
        kaggle_mirror = None
    print(f"\n{'='*60}")
    print(f"Submission: {submission_path}  fingerprint={fingerprint[:12]}  workers={workers}")
    print(f"Output: {run_dir}")
    if kaggle_mirror:
        print(f"Mirror: {kaggle_mirror}")
    print(f"Jobs: {len(jobs)}  timeout 300s each")
    # Load completed
    completed = {}
    pending = []
    problem_root_rel = problem_root
    for sp, rel, idx, prob, timeout in jobs:
        if rel not in completed:
            completed[rel] = _load_completed(_result_path(run_dir, rel))
        if prob["id"] not in completed[rel]:
            pending.append((sp, rel, idx, prob, timeout))
    print(f"Resumed: {len(jobs)-len(pending)}  Pending: {len(pending)}")
    strip_answer = bool(profile["execution"]["strip_answer_before_solver"])
    # Run
    from tqdm.auto import tqdm
    done = len(jobs) - len(pending)
    pbar = tqdm(total=len(jobs), initial=done, desc=submission_name)
    def _run_one(sp, rel, idx, prob, timeout):
        cfg = copy.deepcopy(base_config)
        cfg["solver"]["timeout_seconds"] = timeout
        public = dict(prob)
        if strip_answer:
            public.pop("answer", None)
        t0 = time.monotonic()
        res = run_solver(submission, public, cfg)
        elapsed = time.monotonic() - t0
        expected = prob.get("answer")
        exp_verdict = "true" if expected is True else "false" if expected is False else None
        return {
            "id": prob["id"],
            "source_file": rel.as_posix(),
            "source_index": idx,
            "eq1_id": prob["eq1_id"],
            "eq2_id": prob["eq2_id"],
            "difficulty": prob.get("difficulty"),
            "expected_answer": expected,
            "timeout_seconds": timeout,
            "elapsed_seconds": round(elapsed, 3),
            "label_match": (res.get("verdict") == exp_verdict if res.get("solved") and exp_verdict is not None else None),
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            **res,
        }
    # ThreadPool
    futures = {}
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for sp, rel, idx, prob, timeout in pending:
            fut = ex.submit(_run_one, sp, rel, idx, prob, timeout)
            futures[fut] = (sp, rel, idx, prob)
        for fut in as_completed(futures):
            sp, rel, idx, prob = futures[fut]
            row = fut.result()
            # Append to repo run_dir
            path = _result_path(run_dir, rel)
            _append_result(path, row)
            if kaggle_mirror:
                kpath = kaggle_mirror / rel.parent / f"{rel.name}.results.jsonl"
                _append_result(kpath, row)
            completed[rel][row["id"]] = row
            done += 1
            pbar.update(1)
            state = "SOLVED" if row["solved"] else "FAILED"
            # Throttle prints: every 1 for first 10, then every 5
            if done <= 10 or done % 5 == 0 or not row["solved"]:
                tqdm.write(f"[{done}/{len(jobs)}] {rel}:{row['id']} {state} {row['elapsed_seconds']:.1f}s [judge:{row['judge_calls']} llm:{row['llm_calls']}] label_match={row['label_match']}")
            if done % 25 == 0:
                # Summarize
                from pipeline.solo_benchmark import _summarize
                # Build completed dict keyed by Path
                comp_by_path = {REPO_ROOT / profile["problems"].get("root","") / k if not k.is_absolute() else k: v for k,v in completed.items()}
                # Simpler: call summarize with jobs structured
                # We rebuild jobs dataclass for summary
                from pipeline.solo_benchmark import ProblemJob
                _jobs = [ProblemJob(sp, rel2, idx2, prob2, to) for sp, rel2, idx2, prob2, to in jobs]
                summ = _summarize(_jobs, completed)
                _atomic_write_json(run_dir / "summary.json", summ)
                if kaggle_mirror:
                    _atomic_write_json(kaggle_mirror / "summary.json", summ)
                tqdm.write(f"  checkpoint summary: {summ['correct']}/{summ['completed']} correct  solved={summ['solved']}")
    pbar.close()
    # Final summary
    from pipeline.solo_benchmark import ProblemJob, _summarize
    _jobs = [ProblemJob(sp, rel, idx, prob, to) for sp, rel, idx, prob, to in jobs]
    summary = _summarize(_jobs, completed)
    _atomic_write_json(run_dir / "summary.json", summary)
    if kaggle_mirror:
        _atomic_write_json(kaggle_mirror / "summary.json", summary)
        # Also copy manifest
        if (run_dir / "manifest.json").exists():
            shutil.copy2(run_dir / "manifest.json", kaggle_mirror / "manifest.json")
    print(f"Complete {submission_name}: {summary['correct']}/{summary['completed']} correct ({summary['accuracy']:.2%}) solved={summary['solved']} judge={summary['judge_calls']} llm={summary['llm_calls']} time={summary['elapsed_seconds']:.1f}s")
    if staging:
        staging.cleanup()
    return run_dir, summary

print("Runner defined: run_one_submission()")

In [ ]:
import os; os.chdir(REPO_ROOT)
# ── 4.1 Run FULL benchmark — both submissions sequentially (resume-safe) ──
# Each takes ~1h @ workers=4 (100×300s/4). Total ~1.7h. Kaggle limit 12h → fits.
# Interrupt and re-run → resumes from last completed problem.
import time
from pathlib import Path
PROFILE_PATH = Path("benchmarks/solo_hard_100.json")
ORDER = ["structural", "nhatminh"]  # only 2: structural_cache first, then nhatminh (OpenRouter)
# Override workers here (Kaggle sweet spot 4)
KAGGLE_WORKERS = 4
results = {}
overall_t0 = time.monotonic()
for name in ORDER:
    sub = SUBMISSIONS[name]
    if not sub.exists():
        print(f"Skipping {name}: not found {sub}")
        continue
    print(f"\n{'#'*70}")
    print(f"# Starting {name}: {sub}")
    print(f"{'#'*70}")
    t0 = time.monotonic()
    try:
        run_dir, summary = run_one_submission(sub, PROFILE_PATH, workers=KAGGLE_WORKERS)
        results[name] = (run_dir, summary)
    except Exception as e:
        print(f"FAILED {name}: {e}")
        import traceback; traceback.print_exc()
        results[name] = (None, {"error": str(e)})
    print(f"{name} wall: {(time.monotonic()-t0)/60:.1f} min")
    # Sync to kaggle working after each submission
    if Path("/kaggle/working").exists():
        !cp -r pipeline/results/solo_hard_100 /kaggle/working/ 2>&1 | tail -5
        print("Synced to /kaggle/working")
print(f"\nAll done wall: {(time.monotonic()-overall_t0)/3600:.2f} h")
for k, (d, s) in results.items():
    print(f"{k:12s} {s.get('correct','?')}/{s.get('completed','?')} correct  accuracy={s.get('accuracy','?')}  solved={s.get('solved','?')}  dir={d}")

## 5 — Aggregate + visualize (comparison table)

In [ ]:
import json, pathlib, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# Load all summaries from repo output
out_root = Path("pipeline/results/solo_hard_100")
rows = []
for p in out_root.glob("*/summary.json"):
    s = json.loads(p.read_text())
    name = p.parent.name.split("-")[0]
    rows.append({
        "submission": name,
        "dir": str(p.parent),
        "completed": s["completed"],
        "solved": s["solved"],
        "correct": s["correct"],
        "accuracy": s["accuracy"],
        "label_mismatches": s["label_mismatches"],
        "judge_calls": s["judge_calls"],
        "llm_calls": s["llm_calls"],
        "elapsed_seconds": s["elapsed_seconds"],
    })
df = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
df.style.format({"accuracy": "{:.2%}"})
df

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
# Bar: accuracy per submission
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18,5))
    df_sorted = df.sort_values("accuracy")
    axes[0].barh(df_sorted["submission"], df_sorted["accuracy"])
    axes[0].set_title("Accuracy (correct/completed) — solo_hard_100")
    axes[0].set_xlim(0,1)
    for i, v in enumerate(df_sorted["accuracy"]):
        axes[0].text(v+0.01, i, f"{v:.1%}", va="center")
    axes[1].barh(df_sorted["submission"], df_sorted["solved"])
    axes[1].set_title("Solved (judge accepted)")
    axes[2].barh(df_sorted["submission"], df_sorted["elapsed_seconds"]/3600)
    axes[2].set_title("Aggregate case time (h)")
    plt.tight_layout()
    plt.show()
else:
    print("No summaries yet — run Cell 4.1 first")

# Per-source breakdown for best submission
if not df.empty:
    best = df.iloc[0]["submission"]
    # Find its summary
    import json, pathlib
    from pathlib import Path
    for p in Path("pipeline/results/solo_hard_100").glob(f"{best}-*/summary.json"):
        s = json.loads(p.read_text())
        print(f"Best: {best}  by_source:")
        for src, vals in s["by_source"].items():
            print(f"  {src:40s}  solved {vals['solved']}/{vals['completed']}  time {vals['elapsed_seconds']:.1f}s")
        break

In [ ]:
# ── Detailed per-problem view (optional, for debugging) ──
import json, pathlib, pandas as pd
from pathlib import Path
# Pick one submission to inspect
INSPECT = "solver"  # change to nhatminh/structural/true/typed
out_root = Path("pipeline/results/solo_hard_100")
matches = sorted(out_root.glob(f"{INSPECT}-*"))
if matches:
    run_dir = matches[-1]
    print(f"Inspecting {run_dir}")
    rows = []
    for f in run_dir.rglob("*.results.jsonl"):
        for line in f.read_text().splitlines():
            if line.strip():
                rows.append(json.loads(line))
    dfp = pd.DataFrame(rows)
    if not dfp.empty:
        # Show mismatches first
        dfp["ok"] = dfp["label_match"]
        display(dfp.sort_values(["ok","elapsed_seconds"], ascending=[True, False]).head(20))
        print(f"Total {len(dfp)}  solved {dfp['solved'].sum()}  correct {(dfp['label_match']==True).sum()}  mismatches {(dfp['label_match']==False).sum()}")
    else:
        print("No rows yet")
else:
    print(f"No run dir for {INSPECT} — check Cell 4.1")

## 6 — Export (Kaggle)

Results are under `pipeline/results/solo_hard_100/<name>-<hash>/` and mirrored to `/kaggle/working/solo_hard_100_results/`. Download via Kaggle Output tab or copy to Drive.

In [ ]:
import shutil, pathlib
from pathlib import Path
import json
# Ensure mirrored
if Path("/kaggle/working").exists():
    !mkdir -p /kaggle/working/solo_hard_100_results
    !cp -r pipeline/results/solo_hard_100 /kaggle/working/solo_hard_100_results_full 2>&1 | tail -10
    !ls -lh /kaggle/working/solo_hard_100_results* 2>&1 | head -50
    print("Mirrored to /kaggle/working")
!ls -lh pipeline/results/solo_hard_100 2>&1 | head -50

# Zip for download
!cd /kaggle/working 2>/dev/null && zip -r solo_hard_100_results.zip solo_hard_100_results* 2>&1 | tail -20 || zip -r /tmp/solo_hard_100_results.zip pipeline/results/solo_hard_100 2>&1 | tail -20
!ls -lh /kaggle/working/*.zip 2>&1 | head -10
!ls -lh /tmp/*.zip 2>&1 | head -10

# Also write a combined CSV for quick analysis
import pandas as pd, json, pathlib
from pathlib import Path
rows=[]
for p in Path("pipeline/results/solo_hard_100").rglob("*.results.jsonl"):
    for line in p.read_text().splitlines():
        if line.strip():
            r=json.loads(line)
            r["submission_dir"] = p.parent.name
            rows.append(r)
if rows:
    df_all = pd.DataFrame(rows)
    out_csv = Path("/kaggle/working/solo_hard_100_all.csv") if Path("/kaggle/working").exists() else Path("solo_hard_100_all.csv")
    df_all.to_csv(out_csv, index=False)
    print(f"Wrote {out_csv}  rows={len(df_all)}  cols={list(df_all.columns)[:10]}...")
    df_all.head()
else:
    print("No rows to export yet")

### Re-run / resume

If Kaggle times out, just re-run from **Cell 4.1** — `run_one_submission` resumes via `_load_completed` (`pipeline/solo_benchmark.py:176`). No duplicate rows (`duplicate_policy: exact_id_selection`). To force fresh run, delete the run dir:

```
!rm -rf pipeline/results/solo_hard_100/<name>-<hash>
```

Switch `KAGGLE_WORKERS` to `2` if you hit Lean OOM, or `8` if you have 8 vCPUs.